# CineAI v2 — train on Colab free tier
**Before you start:** copy `cineai.py` and `app.py` to `MyDrive/CineAI/code/`. Runtime → *CPU is enough* (no GPU or High-RAM needed).
Every slow step is checkpointed to Drive, so a disconnect only costs you the current step.

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"          # required by `implicit`; must be set before numpy loads
!pip -q install implicit lightgbm shap fastapi uvicorn
from google.colab import drive; drive.mount("/content/drive")
import sys; ROOT = "/content/drive/MyDrive/CineAI"; sys.path.insert(0, ROOT + "/code")

## 1 · Data → leak-free temporal split
Per user: oldest 75 % = train, next 10 % = val, newest 15 % = test (the model never sees the future).

In [ ]:
import gc, numpy as np, pandas as pd, cineai as ca
DATA = ROOT + "/ml-32m/ml-32m"; ART = ROOT + "/Artifact_v2"; os.makedirs(ART, exist_ok=True)
ratings = ca.load_ratings(DATA + "/ratings.csv")                      # int32/float32 -> ~0.5 GB instead of ~2 GB
APP = ROOT + "/app_interactions.csv"                              # optional: real interactions from your app (backend/export_interactions.py)
if os.path.exists(APP): ratings = pd.concat([ratings, ca.load_ratings(APP)], ignore_index=True); print("merged app interactions:", APP)
movies  = pd.read_csv(DATA + "/movies.csv")
tags = pd.read_csv(DATA + "/tags.csv", usecols=["movieId", "tag"])    # user tags make the content model much better
item_text = tags.groupby("movieId")["tag"].apply(lambda s: " ".join(s.astype(str).str.lower().head(50)))

train, val, test = ca.temporal_split(ratings, val_frac=0.10, test_frac=0.15)
rs = ca.Recsys(movies)
uidx = pd.Index(np.sort(ratings.userId.unique()))
trc, vac, tec = (ca.to_csr(d, uidx, rs.midx) for d in (train, val, test))
del ratings, train, val, test, tags; gc.collect()
print("train/val/test ratings:", trc.nnz, vac.nnz, tec.nnz)

## 2 · Base models (ALS item factors + content embedding + statistics) — slowest step, checkpointed

In [ ]:
BASE = ART + "/rs_base.pkl"
if os.path.exists(BASE): rs = ca.Recsys.load(BASE)
else:
    rs.fit_base(trc, item_text); rs.save(BASE)       # ~10-20 min on free CPU; set cfg als_iters=10 to go faster
print("item factors:", rs.VF.shape)

## 3 · Ranker + taste model
Trained on **val-period labels** (never seen by ALS) with **randomly truncated histories** (1, 2, 3, 5, 8 … ratings), so the ranker learns how much to trust each signal for cold and warm users.
Lower `n_rank` (e.g. 8000) if you run short of time.

In [ ]:
tr, es, used = ca.build_training_sets(rs, trc, vac, n_rank=15000, n_es=3000)
print("training rows:", tr.shape, "| share of rows that are real val ratings:", round((tr.rating > 0).mean(), 3))
rs.fit_rankers(tr, es); rs.save(ART + "/rs_full.pkl")

## 4 · Honest evaluation (test period only)
* **Protocol A – taste:** rank the movies a user *really rated later*; relevant = rating ≥ 4. Compare against the printed base rate (= precision of random guessing).
* **Protocol B – discovery:** rank the whole catalogue; relevant = a ≥ 4 rating in the test period. Absolute numbers here are low for *every* recommender; compare with the ALS / popularity rows.

In [ ]:
rep = ca.evaluate(rs, trc, vac, tec, n_users=2000, exclude=used)
print("Random-guess precision (A):", round(rep["A_base_rate"], 3)); display(rep["A"].round(3))
print("Discovery (B):"); display(rep["B"].round(3)); print(rep["B_extra"])
print("Precision/coverage trade-off of the taste model:"); display(rep["curve"].round(3))
print("Threshold tuned on validation users for 80% precision ->", rep.get("conf80"))

## 5 · Cold-start curve (users truncated to k ratings; ML-32M has no real cold users)

In [ ]:
display(ca.evaluate_cold_start(rs, trc, vac, tec, ks=(0, 1, 3, 5, 10, 20), n_users=1000, exclude=used).round(3))

## 6 · Explainable AI
Exact TreeSHAP comes straight from LightGBM (`pred_contrib=True`), so serving needs no `shap` package.

In [ ]:
import matplotlib.pyplot as plt, shap
X = es.sample(min(20000, len(es)), random_state=0)
c, _ = rs.contributions(X)
shap.summary_plot(c, X[ca.FEATS], show=True)                         # global view
g = pd.DataFrame(rs.grouped(c)).abs().mean().sort_values(); g.plot.barh(title="What drives recommendations (mean |SHAP|)"); plt.show()

def mid_of(t): return int(rs.movies.loc[rs.movies.clean_title.str.contains(t, regex=False), "movieId"].iloc[0])
demo_hist = [(mid_of("Matrix, The"), 5), (mid_of("Inception"), 5), (mid_of("Toy Story"), 4)]   # edit to taste
for r in rs.recommend(demo_hist, top_n=5):
    print(f"{r['title']} ({r['year']})  match {r['match']}%  because of: {r['because']}")
    for x in r["reasons"]: print("   -", x["text"], f"[{x['group']} {x['impact']:+.2f}]")

## 7 · Description (semantic) index — needs a free TMDB key
Fetches overviews for the 20k most-rated movies (resumable), embeds them with a small ONNX model and saves `semantic.npz`.

In [ ]:
import shutil, sys
sys.path.insert(0, ROOT + "/code")            # copy backend/build_semantic.py here, and `!pip -q install fastembed`
import build_semantic as bs
TMDB_KEY = "PASTE_YOUR_TMDB_KEY"
bs.build(rs, item_text, DATA + "/links.csv", TMDB_KEY, ART + "/semantic.npz", cache=ART + "/overviews.json", top=20000)

## 8 · Ship to production
Copy the artefacts next to the docker setup (`./model/`): `rs_full.pkl`, `semantic.npz`, `links.csv`, then `docker compose up -d --build`. See `DEPLOY.md`.

In [ ]:
os.makedirs(ROOT + "/model", exist_ok=True)
for f in (ART + "/rs_full.pkl", ART + "/semantic.npz", DATA + "/links.csv"): shutil.copy(f, ROOT + "/model/")
print(os.listdir(ROOT + "/model"))